# 04 - Guardrails et contrat JSON

Objectif : tester le contrat JSON et les garde-fous.

Fichiers concernés : `src/guardrails.py`, `src/inference.py`, `prompts/json_schema.md`, `outputs/json_contract_check.csv`.

## Décisions de schéma avant pipeline

Décisions officielles pour la suite:
- clé JSON officielle: `visual_evidence`;
- warning officiel: `Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.`;
- classes autorisées: `normal`, `suspected_opacity`, `uncertain`;
- `confidence` doit rester entre 0 et 1;
- le warning est obligatoire;
- la classe `uncertain` est un garde-fou et ne doit pas être supprimée.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = c:\Users\Sarah Efrei\OneDrive\Desktop\2025-2026\S6\MasterCamp\Code du prof\ARVI-RX


In [ ]:
import pandas as pd
from src.guardrails import ALLOWED_CLASSES, REQUIRED_KEYS, WARNING_TEXT, apply_safety_guardrails, validate_prediction
from src.inference import toy_predict

official_keys = {"image_quality", "predicted_class", "confidence", "visual_evidence", "justification", "limitations", "warning"}
print("Classes autorisées:", ALLOWED_CLASSES)
print("Clés officielles:", official_keys)
print("Clés imposées par le code actuel:", REQUIRED_KEYS)
print("Contrat aligné:", official_keys <= REQUIRED_KEYS)
print("Warning officiel:", WARNING_TEXT)

cases = {
    "positive_valid": toy_predict(SAMPLE_IMAGES_DIR / "CXR_SYN_002_suspected_opacity.png"),
    "negative_missing_warning": {"image_quality": "good", "predicted_class": "normal", "confidence": 0.8, "visual_evidence": [], "justification": "test", "limitations": []},
    "negative_bad_class": {"image_quality": "good", "predicted_class": "diagnosis", "confidence": 0.9, "visual_evidence": [], "justification": "test", "limitations": [], "warning": WARNING_TEXT},
    "negative_bad_confidence": {"image_quality": "good", "predicted_class": "normal", "confidence": 1.5, "visual_evidence": [], "justification": "test", "limitations": [], "warning": WARNING_TEXT},
}
rows = []
for name, pred in cases.items():
    before_valid, before_errors = validate_prediction(dict(pred))
    guarded = apply_safety_guardrails(dict(pred))
    after_valid, after_errors = validate_prediction(guarded)
    rows.append({"test_case": name, "valid_before": before_valid, "errors_before": ";".join(before_errors), "predicted_after": guarded.get("predicted_class"), "warning_after": bool(guarded.get("warning")), "valid_after": after_valid, "errors_after": ";".join(after_errors)})
contract_df = pd.DataFrame(rows)
out = OUTPUTS_DIR / "json_contract_check.csv"
contract_df.to_csv(out, index=False)
print(out)
display(contract_df)

Conclusion : le schéma final doit rester aligné sur `visual_evidence`. Les garde-fous
valident le JSON, forcent le warning et ramènent les sorties invalides vers
`uncertain`. Ces règles sont techniques et responsables, mais ne valident pas un
usage médical.